# Supply Chain Twin: Google Colab notebook (data on Google Drive)

Generated by `scripts/build_notebook.py` (at commit `d209912`). The setup cell checks out the
code from GitHub (`REF`, default `main`) into `/content/SupplyTwinAI` and installs it, so Colab
runs exactly the pushed code; the checked-out commit is printed. To change the code, push to
GitHub (or set `REF` to another branch, tag or commit) and re-run the setup cell.

**Google Drive layout** (created on first run; change `DRIVE` in the setup cell to move it):

```
MyDrive/SupplyTwinAI/
  data/raw/provenance.json          dataset (pinned revision), about 110 MB
  data/raw/data/train.parquet
  data/raw/data/eval.parquet        locked final test: integrity-checked, never loaded
  runs/                             all run outputs (metrics, predictions, posteriors)
```

The dataset is read from Drive and copied to the Colab disk for fast reads. If it is not on
Drive yet, the pinned revision is downloaded once and saved there. You can also upload a
local `data/raw/` folder (with `provenance.json`) to that place yourself.

**How to use:** run *Setup*, *Helpers* and *Data* (Runtime → Run all is fine), then the steps
you need. Finished steps keep their outputs on Drive, so you can skip them later.
`FAST = True` (in the helpers cell) keeps every step to a few minutes; set `FAST = False` for
the full runs. If Colab restarts the session, re-run *Setup* and *Helpers* first.
Use a CPU runtime with many cores: the installed JAX is CPU-only, so a GPU is not used.

Rules that every step follows are in `CLAUDE.md` of the repository: rolling-origin
backtesting without leakage, point-in-time features, the official `eval` split is never read.
Sales are normalized units; reconstructed demand is never presented as observed.

## Setup

In [ ]:
import os, sys, pathlib, shutil, subprocess
from google.colab import drive

REPO = "https://github.com/moghbaie/SupplyTwinAI.git"
REF = "main"  # branch, tag or commit to run
DRIVE = pathlib.Path("/content/drive/MyDrive/SupplyTwinAI")  # dataset and run outputs live here
ROOT = pathlib.Path("/content/SupplyTwinAI")  # code checkout (Colab disk, wiped per runtime)

def sh(cmd):
    """Run a shell command in ROOT; stop the cell if it fails."""
    subprocess.run(cmd, shell=True, check=True, cwd=ROOT)

drive.mount("/content/drive")
for d in (DRIVE / "data" / "raw", DRIVE / "runs", ROOT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

# Check out the code (works in an empty folder and over leftovers of an older notebook;
# untracked files such as data/ and the runs link are kept, tracked files are overwritten).
if not (ROOT / ".git").exists():
    sh(f"git init -q && git remote add origin {REPO}")
sh(f"git fetch -q --depth 1 origin {REF} && git checkout -q --force FETCH_HEAD")
sh('git log -1 --format="code: commit %h (%cd) %s" --date=short')

if not os.path.islink("runs"):  # run outputs go straight to Drive
    if os.path.isdir("runs"):
        os.rmdir("runs")  # only succeeds if empty, so nothing is lost
    os.symlink(DRIVE / "runs", "runs")

# Install with all extras (no --force-reinstall: it would upgrade Colab's numpy and force a
# session restart). The import check runs in a fresh process, like the run cells below.
sh('pip install -q -e ".[dev,models,bayes]"')
sh('python -c "import sctwin.censoring, sctwin.demand_model; print(\'package import OK\')"')
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))  # make the package importable in this kernel too
print("code:", ROOT, "| data and runs:", DRIVE, "| CPU cores:", os.cpu_count())

## Helpers

In [ ]:
import copy, glob, json
import pandas as pd
import yaml

os.chdir(ROOT)

FAST = True          # small, quick runs; set False for the full runs
CORES = os.cpu_count() or 2

def config_with(path, updates, name):
    """Copy a YAML config with nested updates applied; returns the new path."""
    cfg = yaml.safe_load(open(path, encoding="utf-8"))
    def merge(a, b):
        for k, v in b.items():
            if isinstance(v, dict) and isinstance(a.get(k), dict):
                merge(a[k], v)
            else:
                a[k] = v
    merge(cfg, copy.deepcopy(updates))
    out = f"configs/nb_{name}.yaml"
    yaml.safe_dump(cfg, open(out, "w", encoding="utf-8"), sort_keys=False)
    return out

def latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(pattern)
    print("reading", files[-1])
    return json.load(open(files[-1], encoding="utf-8"))

def model_table(models):
    rows = {}
    for m, r in models.items():
        iv = r.get("intervals", {})
        rows[m] = {"mae": r.get("mae", r.get("mae_median")), "bias": r.get("bias"),
                   "wis": r.get("wis"), "cov50": iv.get("0.5", {}).get("coverage"),
                   "cov90": iv.get("0.9", {}).get("coverage")}
    return pd.DataFrame(rows).T.round(4)

## Run: tests (synthetic, no data needed)
The leakage tests run for both units, "day" and "hour". Warnings are errors, except
deprecation warnings: Colab's preinstalled libraries raise them inside each other (e.g.
matplotlib calling deprecated pyparsing), which is not this project's code. Local runs and CI
keep plain `-W error`.

In [ ]:
!python -m pytest -W error -W "default::DeprecationWarning" -m "not data" -q -p no:cacheprovider

## Data from Google Drive (run before any step)
Uses the dataset on Drive if present (checksums and pinned revision verified); otherwise
downloads the pinned revision once and saves it to Drive. Copies it to the Colab disk.

In [ ]:
import json, yaml
from sctwin.data import verify_checksums

os.chdir(ROOT)
raw_drive = DRIVE / "data" / "raw"
pinned = yaml.safe_load(open("configs/small.yaml"))["dataset"]["revision"]
skip_cache = shutil.ignore_patterns(".cache")
if (raw_drive / "provenance.json").exists():
    print("dataset found on Drive:", raw_drive)
    shutil.copytree(raw_drive, "data/raw", dirs_exist_ok=True, ignore=skip_cache)
else:
    print("dataset not on Drive yet: downloading the pinned revision once")
    !python scripts/fetch_data.py
    shutil.copytree("data/raw", raw_drive, dirs_exist_ok=True, ignore=skip_cache)
    print("saved to Drive:", raw_drive)
prov = json.load(open("data/raw/provenance.json"))
assert prov["revision"] == pinned, f"Drive has revision {prov['revision']}, config pins {pinned}"
checks = verify_checksums("data/raw")
assert all(checks.values()), f"checksum mismatch: {checks}"
print("revision", pinned[:12], "checksums OK:", checks)

## Run: Step 1, audit and real-data tests
Re-runs the audit (it re-writes the identical split manifest) and the real-data tests,
including the real-data leakage checks.

In [ ]:
!python scripts/audit_data.py
!python -m pytest -m data -q -p no:cacheprovider

## Run: Step 3, baseline backtest (observed sales)
Rolling origin; train on feature times t <= T - L - H; predict (T + L, T + L + H].
Full day run: about 15 min on 12 cores and about 6 GB RAM. FAST uses 200 series and 2 folds.

In [ ]:
cfg3 = config_with("configs/baselines.yaml", {"lgbm": {"num_threads": CORES}}, "baselines")
smoke = "--smoke" if FAST else ""
!python scripts/run_baselines.py --unit day --config {cfg3} {smoke}
!python scripts/run_baselines.py --unit hour --config {cfg3} {smoke}

In [ ]:
for unit in ("day", "hour"):
    m = latest(f"runs/step3*/{unit}/*/metrics.json")
    print(unit, "origins", m["protocol"]["origins"][:5], "...")
    display(model_table(m["metrics"]["overall"]["models"]))
    display(pd.DataFrame({o: {k: v.get("wis", v.get("mae")) for k, v in f["models"].items()}
                          for o, f in m["metrics"]["folds"].items()}).T.round(4))

## Run: Step 4, censoring benchmark (proxy truth)
Demand-driven stockouts injected into fully in-stock days; calibration on the fit window only.
Full run: about 1.5 min for 5,000 series. FAST uses 300 series.

In [ ]:
smoke = "--smoke" if FAST else ""
!python scripts/run_censoring_benchmark.py {smoke}

In [ ]:
m = latest("runs/step4*/*/metrics.json")
print("calibrated injection:", m["calibrated_config"])
display(pd.DataFrame({"real": m["real_fit_window"]["stats"],
                      "injected": m["injected"]["stats"]}).round(3))
display(pd.DataFrame({"real": m["real_fit_window"]["position_ratios"],
                      "injected": m["injected"]["position_ratios"]}).round(3))
print("daily totals on stockout days")
display(model_table(m["benchmark"]["daily"]["overall"]["models"]))

## Run: Step 5, Bayesian model ladder (M0, M1, M2, M1d, M2d)
NumPyro NUTS. FAST: 4 series and short chains (a few minutes). Full: 20 series, 4 chains x
(1000 + 1000) draws; chains run in parallel, one per CPU core (up to 4), so it takes hours on a
2-core runtime. `--part real` and `--part recovery` split the run in two.
M3 = M2 with `--n-series 100 --models M2`. M1d/M2d are exploratory (added after the plan).

In [ ]:
chains = min(4, CORES)
cfg5 = config_with("configs/bayes.yaml", {"mcmc": {"num_chains": chains}}, "bayes")
smoke = "--smoke" if FAST else ""
!python scripts/run_bayes.py --config {cfg5} --part all {smoke}

In [ ]:
m = latest("runs/step5*/*/metrics_all.json")
real = m["real"]
keys = ("runtime_s", "divergences", "max_rhat", "min_ess")
display(pd.DataFrame({k: {x: v["diagnostics"][x] for x in keys}
                      for k, v in real["models"].items()}).T)
print("daily totals on stockout days (proxy truth); baselines are point estimates")
display(model_table(real["daily"]["overall"]["models"]))
print("PSIS-LOO on in-stock hours:", real["loo_compare"])
for name, r in m["recovery"]["models"].items():
    print("recovery", name, r["mean_window_demand"])
    display(pd.DataFrame(r["scalars"]).T)

## Run: Step 5, simulation-based calibration of M1
Ranks of the true parameters among posterior draws should be uniform (chi-square p-values not
small after Holm). Full: 200 replications (about 40 min). FAST: 20.

In [ ]:
reps = 20 if FAST else 200
!python scripts/run_sbc.py --reps {reps}
s = latest("runs/sbc/*/ranks.json")
display(pd.DataFrame(s["scalars"]).T)
print("min Holm-adjusted p:", s["min_p_holm"])
print("divergent replications:", s["divergent_reps"])